# CFPB complaint classifier: Colab training
Runs `cfpb_mlops.training.train` from the GitHub repo against `complaints.csv` stored in your Google Drive.
Set the runtime to **High-RAM** if you can (Runtime > Change runtime type). Run the cells top to bottom.

In [ ]:
# 1. Settings
DRIVE_CSV = "/content/drive/MyDrive/cfpb/complaints.csv"  # where you uploaded the file
SAMPLE_N = 500_000   # most recent N rows with a narrative; None = all ~1.9M (needs lots of RAM)
MAX_FEATURES = 10_000
C = 1.0
TEST_MONTHS = 6
REGISTER = False

In [ ]:
# 2. Mount Drive, clone repo, copy the CSV onto local disk (much faster reads than Drive)
from google.colab import drive
drive.mount("/content/drive")

!rm -rf /content/CFPB-Complaint-Routing
!git clone -q https://github.com/Reehan09Sarmah/CFPB-Complaint-Routing.git /content/CFPB-Complaint-Routing
!mkdir -p /content/CFPB-Complaint-Routing/data
!cp "$DRIVE_CSV" /content/CFPB-Complaint-Routing/data/complaints.csv
!ls -lh /content/CFPB-Complaint-Routing/data
!pip -q install mlflow tqdm

In [ ]:
# 3. Train (calls train() directly because the CLI cannot pass sample_n=None)
import os, sys
sys.path.insert(0, "/content/CFPB-Complaint-Routing/src")

import mlflow
from cfpb_mlops.training.train import train

os.chdir("/content")  # sqlite store + ./mlartifacts land here
mlflow.set_tracking_uri("sqlite:////content/mlflow.db")
train(
    sample_n=SAMPLE_N,
    max_features=MAX_FEATURES,
    C=C,
    test_months=TEST_MONTHS,
    register_model=REGISTER,
)

In [ ]:
# 4. Show the logged metrics, then save the MLflow store to Drive and download it
runs = mlflow.search_runs(experiment_names=["cfpb-complaint-classifier"])
print(runs[["run_id", "params.sample_n", "metrics.accuracy", "metrics.f1_macro"]].to_string())

!cd /content && zip -qr mlruns.zip mlflow.db mlruns mlartifacts
!cp /content/mlruns.zip /content/drive/MyDrive/cfpb/mlruns.zip
from google.colab import files
files.download("/content/mlruns.zip")